In [1]:
from pathlib import Path
import re
import pandas as pd

STAGE1_ROOT = Path("/projects/ovcare/users/nikolay_alabi/immuno/stage1_univariate_v6/results")

FEATURE_SOURCES = [
    "phenotype_only",
    "AR_state",
    "AR_checkpoint_state",
    "compartment",
    "compartment_state",
]

def infer_context_from_path(path: Path):
    parts = list(path.parts)
    ctx = {}
    
    fs_indices = [i for i, p in enumerate(parts) if p in FEATURE_SOURCES]
    if fs_indices:
        i = fs_indices[-1]
        names = ["feature_source", "panel", "feature_group", "cohort", "endpoint", "sample_type"]
        for j, name in enumerate(names):
            k = i + j
            if k < len(parts):
                ctx[name] = parts[k]
        for p in parts[i + 1:]:
            if p.startswith("agg-"):
                ctx["agg"] = p.replace("agg-", "", 1)
            elif p.startswith("patient_subset-"):
                ctx["patient_subset"] = p.replace("patient_subset-", "", 1)
            elif p.startswith("transform-"):
                ctx["transform_mode"] = p.replace("transform-", "", 1)

    suffix_match = re.search(r"__(summary|fullmodels|feature_filter|failures|folds|oof)\.csv$", path.name)
    ctx["suffix"] = suffix_match.group(1) if suffix_match else None
    ctx["path"] = str(path)
    return ctx

files = []
for suffix in ["summary", "fullmodels", "feature_filter"]:
    for fp in STAGE1_ROOT.rglob(f"*__{suffix}.csv"):
        files.append(infer_context_from_path(fp))

files_df = pd.DataFrame(files)

print("Total files:", files_df.shape)
display(
    files_df.groupby(
        ["suffix", "sample_type", "agg", "transform_mode"],
        dropna=False
    )
    .size()
    .reset_index(name="n_files")
    .sort_values(["suffix", "sample_type", "agg", "transform_mode"])
)

display(
    files_df.groupby(
        ["cohort", "panel", "endpoint", "sample_type", "patient_subset", "agg", "transform_mode", "suffix"],
        dropna=False
    )
    .size()
    .reset_index(name="n_files")
    .sort_values(["cohort", "panel", "endpoint", "sample_type", "patient_subset", "agg", "transform_mode", "suffix"])
    .head(100)
)

Total files: (3240, 11)


,suffix,sample_type,agg,transform_mode,n_files
0,feature_filter,RC,median,zscore,472
1,feature_filter,TURBT,median,zscore,608
2,fullmodels,RC,median,zscore,472
3,fullmodels,TURBT,median,zscore,608
4,summary,RC,median,zscore,472
5,summary,TURBT,median,zscore,608


,cohort,panel,endpoint,sample_type,patient_subset,agg,transform_mode,suffix,n_files
0,BLASST,AR,OS,TURBT,all,median,zscore,feature_filter,20
1,BLASST,AR,OS,TURBT,all,median,zscore,fullmodels,20
2,BLASST,AR,OS,TURBT,all,median,zscore,summary,20
3,BLASST,AR,RFS,RC,all,median,zscore,feature_filter,20
4,BLASST,AR,RFS,RC,all,median,zscore,fullmodels,20
...,...,...,...,...,...,...,...,...,...
95,NAC2015,AR,OS,TURBT,all,median,zscore,summary,20
96,NAC2015,AR,RFS,RC,all,median,zscore,feature_filter,20
97,NAC2015,AR,RFS,RC,all,median,zscore,fullmodels,20
98,NAC2015,AR,RFS,RC,all,median,zscore,summary,20


In [2]:
# Edit this block depending on the run you want to feed into stage2_module_pipeline_v7.py

REQUESTED = {
    "cohorts": ["NAC2020", "PURE01", "BLASST", "No-NAC", "NAC2015", "KOLL"],
    "panels": ["AR", "BT"],
    "endpoints": ["complete_response", "any_response", "OS", "RFS"],
    "sample_types": ["TURBT", "RC"],
    "patient_subsets": ["all", "no_adj_chemo"],
    "aggs": ["median"],
    "transform_modes": ["zscore", "log1p_zscore"],
}

x = files_df.copy()

for col, vals in REQUESTED.items():
    col = col[:-1] if col.endswith("s") else col
    if col in x.columns:
        x = x[x[col].astype(str).isin(vals)].copy()

# No-NAC and KOLL response contexts are not expected
bad_response = (
    x["cohort"].astype(str).isin(["No-NAC", "KOLL"])
    & x["endpoint"].astype(str).isin(["complete_response", "any_response"])
)
x = x[~bad_response].copy()

presence = (
    x.groupby(
        ["cohort", "panel", "feature_source", "feature_group", "endpoint",
         "sample_type", "patient_subset", "agg", "transform_mode", "suffix"],
        dropna=False
    )
    .size()
    .reset_index(name="n_files")
)

wide = (
    presence.pivot_table(
        index=["cohort", "panel", "feature_source", "feature_group", "endpoint",
               "sample_type", "patient_subset", "agg", "transform_mode"],
        columns="suffix",
        values="n_files",
        fill_value=0,
        aggfunc="sum"
    )
    .reset_index()
)

for c in ["summary", "fullmodels", "feature_filter"]:
    if c not in wide.columns:
        wide[c] = 0

wide["has_required_stage1_outputs"] = (
    (wide["summary"] > 0)
    & (wide["fullmodels"] > 0)
    & (wide["feature_filter"] > 0)
)

print("Requested context-transform rows:", wide.shape[0])
print("Rows with all required outputs:", int(wide["has_required_stage1_outputs"].sum()))
print("Rows missing something:", int((~wide["has_required_stage1_outputs"]).sum()))

display(
    wide.groupby(["sample_type", "agg", "transform_mode"])["has_required_stage1_outputs"]
    .agg(["sum", "count"])
    .reset_index()
)

display(
    wide[~wide["has_required_stage1_outputs"]]
    .sort_values(["cohort", "panel", "feature_source", "feature_group", "endpoint",
                  "sample_type", "patient_subset", "agg", "transform_mode"])
    .head(100)
)

Requested context-transform rows: 1080
Rows with all required outputs: 1080
Rows missing something: 0


,sample_type,agg,transform_mode,sum,count
0,RC,median,zscore,472,472
1,TURBT,median,zscore,608,608


suffix,cohort,panel,feature_source,feature_group,endpoint,sample_type,patient_subset,agg,transform_mode,feature_filter,fullmodels,summary,has_required_stage1_outputs


In [3]:
paired = wide[wide["has_required_stage1_outputs"]].copy()

pair_index = [
    "cohort", "panel", "feature_source", "feature_group", "endpoint",
    "sample_type", "patient_subset", "agg"
]

pair_summary = (
    paired.groupby(pair_index, dropna=False)["transform_mode"]
    .agg(lambda x: ";".join(sorted(set(x.astype(str)))))
    .reset_index(name="available_transform_modes")
)

pair_summary["has_zscore"] = pair_summary["available_transform_modes"].str.contains("zscore")
pair_summary["has_log1p_zscore"] = pair_summary["available_transform_modes"].str.contains("log1p_zscore")
pair_summary["has_both_transforms"] = pair_summary["has_zscore"] & pair_summary["has_log1p_zscore"]

display(
    pair_summary.groupby(["sample_type", "agg"])["has_both_transforms"]
    .agg(["sum", "count"])
    .reset_index()
)

display(
    pair_summary[~pair_summary["has_both_transforms"]]
    .sort_values(pair_index)
    .head(100)
)

,sample_type,agg,sum,count
0,RC,median,0,472
1,TURBT,median,0,608


,cohort,panel,feature_source,feature_group,endpoint,sample_type,patient_subset,agg,available_transform_modes,has_zscore,has_log1p_zscore,has_both_transforms
0,BLASST,AR,AR_checkpoint_state,NN,OS,TURBT,all,median,zscore,True,False,False
1,BLASST,AR,AR_checkpoint_state,NN,RFS,RC,all,median,zscore,True,False,False
2,BLASST,AR,AR_checkpoint_state,NN,RFS,TURBT,all,median,zscore,True,False,False
3,BLASST,AR,AR_checkpoint_state,NN,any_response,RC,all,median,zscore,True,False,False
4,BLASST,AR,AR_checkpoint_state,NN,any_response,TURBT,all,median,zscore,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
95,BLASST,AR,compartment_state,athena,any_response,TURBT,all,median,zscore,True,False,False
96,BLASST,AR,compartment_state,athena,complete_response,RC,all,median,zscore,True,False,False
97,BLASST,AR,compartment_state,athena,complete_response,TURBT,all,median,zscore,True,False,False
98,BLASST,AR,compartment_state,cell_features,OS,TURBT,all,median,zscore,True,False,False
